<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l8.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L8 · Grid search + sizing
12 combinaciones (fast × slow) con costos, y tamaño por Kelly capado: buscar edge y sobrevivirle.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c2_l8_grid.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/python/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
best = df.loc[df["sharpe"].idxmax()]
print(f"mejor grid: ({int(best['fast'])},{int(best['slow'])})  Sharpe={best['sharpe']:.3f}  MaxDD={best['max_dd']:.2%}  f_kelly={best['f_kelly']:.3f}")
print(df.sort_values("sharpe", ascending=False).head(5).to_string(index=False))

## Kelly dice cuánto, el drawdown dice hasta cuándo
Kelly sin capar pide tamaños suicidas; aquí va capado a 0.25. Regla: Sharpe > 1,0 para el sello robusta.

In [ ]:
import matplotlib.pyplot as plt

color = {"robusta": "#5eead4", "debil": "#8a8a93"}
fig, ax = plt.subplots(figsize=(8, 4))
for v, g in df.groupby("veredicto"):
    ax.scatter(g["max_dd"]*100, g["sharpe"], c=color[v], label=v, s=90)
for _, r in df.iterrows():
    ax.text(r["max_dd"]*100, r["sharpe"], f"({int(r['fast'])},{int(r['slow'])})", fontsize=7)
ax.set_xlabel("MaxDD (%)")
ax.set_ylabel("Sharpe")
ax.set_title("Grid: Sharpe vs drawdown (tamaño ∝ Kelly)")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Chequeos automáticos
assert len(df) == 12, "se esperan 12 combinaciones"
assert ((df["veredicto"] == "robusta") == (df["sharpe"] > 1.0)).all(), "robusta = Sharpe > 1.0"
assert (df["f_objetivo"] <= 1.0).all() and (df["f_objetivo"] >= 0).all(), "tamaño objetivo en [0, 1]"
assert (df["f_kelly"] > 1).any(), "Kelly sin capar pide más del 100%: absurdo operativo"
assert int(best['fast']) == 5 and int(best['slow']) == 50, "mejor grid determinista"
assert (df['veredicto'] == 'robusta').sum() == 7, "nº de robustas determinista"
print("OK: grid + sizing verificados")